In [1062]:
import pandas
import numpy
import re
import nltk
from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords
# from nltk.stem import WordNetLemmatizer
from nltk.stem import PorterStemmer # disadvantage : Correct root word is not returned,only removes the tail end
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split
from sklearn.naive_bayes import MultinomialNB # The multinomial Naive Bayes classifier is suitable for classification with discrete
                                                # features (e.g., word counts for text classification). The multinomial distribution 
                                                # normally requires integer feature counts. However, in practice, fractional counts such as tf-idf may also work.
from sklearn.naive_bayes import GaussianNB
from sklearn.metrics import accuracy_score

In [1063]:
data = pandas.read_csv("E-Commerce.csv")
df = pandas.DataFrame(data)
data

,text,intent
0,"Where is my order???,",track_order
1,track my order pls,track_order
2,"order status,",track_order
3,"can u track my package,",track_order
4,"my order is not showing,",track_order
...,...,...
57,"payment error occurred,",payment_issue
58,"transaction unsuccessful,",payment_issue
59,"payment failed again,",payment_issue
60,"amount deited ut no order,",payment_issue


In [1064]:
df.isna().sum()

text      6
intent    2
dtype: int64

has NaN values
imbalanced data
extra punctuation
repeated words
informal language
missing values

In [1065]:
df = df.dropna(subset = ["text","intent"])

In [1066]:
df.isna().sum()

text      0
intent    0
dtype: int64

In [1067]:
df["text"] = df["text"].str.lower()

In [1068]:
def replace_punctuation(text):
    text = re.sub(r'http\S+'," ",text)
    text = re.sub(r'[^a-z\s]'," ",text)
    return text
df["text"] = df["text"].apply(replace_punctuation)

In [1069]:
df

,text,intent
0,where is my order,track_order
1,track my order pls,track_order
2,order status,track_order
3,can u track my package,track_order
4,my order is not showing,track_order
5,track order now,track_order
6,order shipped or not,track_order
7,status of my order,track_order
8,order id track,track_order
9,where s my order,track_order


In [1070]:
df["text"] = df["text"].apply(word_tokenize)

In [1071]:
df

,text,intent
0,"[where, is, my, order]",track_order
1,"[track, my, order, pls]",track_order
2,"[order, status]",track_order
3,"[can, u, track, my, package]",track_order
4,"[my, order, is, not, showing]",track_order
5,"[track, order, now]",track_order
6,"[order, shipped, or, not]",track_order
7,"[status, of, my, order]",track_order
8,"[order, id, track]",track_order
9,"[where, s, my, order]",track_order


In [1072]:
stop_words = set(stopwords.words("english"))

In [1073]:
df["text"] = df["text"].apply(lambda x : [i for i in x if i not in stop_words] )

In [1074]:
df

,text,intent
0,[order],track_order
1,"[track, order, pls]",track_order
2,"[order, status]",track_order
3,"[u, track, package]",track_order
4,"[order, showing]",track_order
5,"[track, order]",track_order
6,"[order, shipped]",track_order
7,"[status, order]",track_order
8,"[order, id, track]",track_order
9,[order],track_order


In [1075]:
stemmer = PorterStemmer()
df["text"] = df["text"].apply(lambda x : [stemmer.stem(i) for i in x])

In [1076]:
df["text"] = df["text"].apply(lambda x:" ".join(x))

In [1077]:
vectorizer = TfidfVectorizer(
    ngram_range=(1, 2),
    min_df=1
)
X = tfid.fit_transform(df["text"])

In [1078]:
X

<Compressed Sparse Row sparse matrix of dtype 'float64'
	with 219 stored elements and shape (56, 112)>

In [1079]:
from sklearn.preprocessing import LabelEncoder
le = LabelEncoder()
Y = le.fit_transform(df['intent'])

In [1080]:
Y

array([4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 0, 0, 0, 0, 0, 0, 0, 0, 0,
       0, 0, 0, 3, 3, 3, 3, 3, 3, 3, 3, 3, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
       1, 1, 1, 1, 1, 2, 2, 2, 2, 2, 2, 2])

In [1081]:
from imblearn.over_sampling import SMOTE
sm = SMOTE(random_state = 42)
x_new,y_new = sm.fit_resample(X,Y)
pandas.DataFrame(y_new).value_counts()

0
0    15
1    15
2    15
3    15
4    15
Name: count, dtype: int64

In [1082]:
X_train,X_test,Y_train,Y_test = train_test_split(X,Y,test_size = 0.3)

In [1083]:
print(Y_test)
print(y_pred)

[1 0 1 4 3 2 3 1 0 2 4 3 0 2 1 0 1]
[4 1 0 4 1 4 1 4 4 0 0 3 1 4 1 4 3]


In [1084]:
obj = MultinomialNB()
model = obj.fit(X_train,Y_train)
y_pred = model.predict(X_test)
print(f"Accuracy Score is {accuracy_score(Y_test,y_pred)}")

Accuracy Score is 0.7058823529411765


In [1085]:
from sklearn.metrics import classification_report

print(classification_report(Y_test, y_pred))


              precision    recall  f1-score   support

           0       1.00      1.00      1.00         4
           1       0.83      1.00      0.91         5
           2       1.00      0.33      0.50         3
           3       0.00      0.00      0.00         3
           4       0.33      1.00      0.50         2

    accuracy                           0.71        17
   macro avg       0.63      0.67      0.58        17
weighted avg       0.70      0.71      0.65        17



C:\Users\ASUS\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\LocalCache\local-packages\Python311\site-packages\sklearn\metrics\_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
C:\Users\ASUS\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\LocalCache\local-packages\Python311\site-packages\sklearn\metrics\_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
C:\Users\ASUS\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\LocalCache\local-packages\Python311\site-packages\sklearn\metrics\_classif

In [1086]:
from sklearn.metrics import confusion_matrix
cm = confusion_matrix(Y_test, y_pred)
cm


array([[4, 0, 0, 0, 0],
       [0, 5, 0, 0, 0],
       [0, 0, 1, 0, 2],
       [0, 1, 0, 0, 2],
       [0, 0, 0, 0, 2]], dtype=int64)

In [1087]:
import joblib

joblib.dump(model, "intent_model.pkl")
joblib.dump(tfid, "tfidf_vectorizer.pkl")
joblib.dump(le, "label_encoder.pkl")



['label_encoder.pkl']